# Pertemuan 10 — Pemrosesan Data Streaming dengan Spark Structured Streaming (Main Lab)
**Praktikan:** Sholahuddin Ahmad (NPM: 250506070)

### 1. Inisialisasi SparkSession & Folder Simulasi

In [1]:
from spark_init import spark, F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
import os
import shutil
import time
import pandas as pd
from datetime import datetime

spark.conf.set("spark.sql.shuffle.partitions", "4")

folder_stream = "simulasi_stream_transaksi"
folder_checkpoint = "checkpoint_pertemuan10"

shutil.rmtree(folder_stream, ignore_errors=True)
shutil.rmtree(folder_checkpoint, ignore_errors=True)
os.makedirs(folder_stream, exist_ok=True)

print(f"⚡ SparkSession aktif. Folder streaming '{folder_stream}' siap.")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/08 08:01:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/11/08 08:02:03 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


⚡ SparkSession siap! (Spark 3.5.9 | local[*])


⚡ SparkSession aktif. Folder streaming 'simulasi_stream_transaksi' siap.


### 2. Definisi Skema Eksplisit (StructType)

In [2]:
skema_transaksi = StructType([
    StructField("trx_id", StringType(), True),
    StructField("kota", StringType(), True),
    StructField("kategori", StringType(), True),
    StructField("pendapatan", IntegerType(), True),
])

print("Skema transaksi terdefinisi:")
print(skema_transaksi)

Skema transaksi terdefinisi:
StructType([StructField('trx_id', StringType(), True), StructField('kota', StringType(), True), StructField('kategori', StringType(), True), StructField('pendapatan', IntegerType(), True)])


### 3. Streaming Query Agregasi Kota (Complete Mode & Memory Sink)

In [3]:
for q in spark.streams.active:
    if q.name == "hasil_agregasi":
        q.stop()

df_stream = spark.readStream \
    .format("csv") \
    .schema(skema_transaksi) \
    .option("header", True) \
    .load(folder_stream)

agregasi_stream = df_stream.groupBy("kota").agg(
    F.sum("pendapatan").alias("total_pendapatan"),
    F.count("trx_id").alias("jumlah_transaksi")
)

query1 = agregasi_stream.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("hasil_agregasi") \
    .start()

print("Status query 1:", query1.status)

Status query 1: {'message': 'Getting offsets from FileStreamSource[file:/mnt/alterra/repos/praktikum-big-data/pertemuan-10/simulasi_stream_transaksi]', 'isDataAvailable': False, 'isTriggerActive': True}


### 4. Ingestion Data Batch 1

In [4]:
pd.DataFrame({
    "trx_id": ["T1", "T2", "T3"],
    "kota": ["Magelang", "Solo", "Magelang"],
    "kategori": ["Elektronik", "Fashion", "Makanan"],
    "pendapatan": [150000, 200000, 75000],
}).to_csv(f"{folder_stream}/batch_1.csv", index=False)

time.sleep(2)
query1.processAllAvailable()

spark.sql("SELECT * FROM hasil_agregasi ORDER BY total_pendapatan DESC").show()

+--------+----------------+----------------+
|    kota|total_pendapatan|jumlah_transaksi|
+--------+----------------+----------------+
|Magelang|          225000|               2|
|    Solo|          200000|               1|
+--------+----------------+----------------+



### 5. Ingestion Data Batch 2

In [5]:
pd.DataFrame({
    "trx_id": ["T4", "T5"],
    "kota": ["Magelang", "Yogyakarta"],
    "kategori": ["Fashion", "Elektronik"],
    "pendapatan": [300000, 500000],
}).to_csv(f"{folder_stream}/batch_2.csv", index=False)

time.sleep(2)
query1.processAllAvailable()

spark.sql("SELECT * FROM hasil_agregasi ORDER BY total_pendapatan DESC").show()

+----------+----------------+----------------+
|      kota|total_pendapatan|jumlah_transaksi|
+----------+----------------+----------------+
|  Magelang|          525000|               3|
|Yogyakarta|          500000|               1|
|      Solo|          200000|               1|
+----------+----------------+----------------+



### 6. Penghentian Streaming Query 1

In [6]:
query1.stop()
print("Streaming query 1 dihentikan. Status aktif:", query1.isActive)

Streaming query 1 dihentikan. Status aktif: False


### 7. Streaming Query Window Waktu (1 Menit)

In [7]:
for q in spark.streams.active:
    if q.name == "hasil_window":
        q.stop()

skema_dengan_waktu = StructType([
    StructField("trx_id", StringType(), True),
    StructField("kota", StringType(), True),
    StructField("pendapatan", IntegerType(), True),
    StructField("waktu_transaksi", TimestampType(), True),
])

folder_stream2 = "simulasi_stream_waktu"
shutil.rmtree(folder_stream2, ignore_errors=True)
os.makedirs(folder_stream2, exist_ok=True)

df_stream2 = spark.readStream \
    .format("csv") \
    .schema(skema_dengan_waktu) \
    .option("header", True) \
    .load(folder_stream2)

agregasi_window = df_stream2.groupBy(
    F.window(F.col("waktu_transaksi"), "1 minute")
).agg(
    F.sum("pendapatan").alias("total_pendapatan"),
    F.count("trx_id").alias("jumlah_transaksi")
)

query2 = agregasi_window.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("hasil_window") \
    .start()

print("Streaming query window waktu aktif.")

Streaming query window waktu aktif.


### 8. Ingestion Data Timestamp Window

In [8]:
waktu_sekarang = datetime.now()
pd.DataFrame({
    "trx_id": ["W1", "W2"],
    "kota": ["Semarang", "Solo"],
    "pendapatan": [120000, 180000],
    "waktu_transaksi": [waktu_sekarang.strftime("%Y-%m-%d %H:%M:%S")] * 2,
}).to_csv(f"{folder_stream2}/batch_w1.csv", index=False)

time.sleep(2)
query2.processAllAvailable()

spark.sql("SELECT window.start, window.end, total_pendapatan, jumlah_transaksi FROM hasil_window").show(truncate=False)

+-------------------+-------------------+----------------+----------------+
|start              |end                |total_pendapatan|jumlah_transaksi|
+-------------------+-------------------+----------------+----------------+
|2026-11-08 08:02:00|2026-11-08 08:03:00|300000          |2               |
+-------------------+-------------------+----------------+----------------+



### 9. Penghentian Streaming Query 2

In [9]:
query2.stop()
print("Query 2 dihentikan. Status aktif:", query2.isActive)

Query 2 dihentikan. Status aktif: False


### Latihan Mandiri — Persiapan Lingkungan

In [10]:
folder_latihan = "latihan10_stream"
shutil.rmtree(folder_latihan, ignore_errors=True)
os.makedirs(folder_latihan, exist_ok=True)

skema_latihan = StructType([
    StructField("trx_id", StringType(), True),
    StructField("kategori", StringType(), True),
    StructField("pendapatan", IntegerType(), True),
])

print("Folder latihan siap:", folder_latihan)

Folder latihan siap: latihan10_stream


### Latihan Mandiri — Soal 1: Streaming Query Rata-Rata Pendapatan per Kategori

In [11]:
for q in spark.streams.active:
    if q.name == "latihan_agg":
        q.stop()

df_stream_latihan = spark.readStream \
    .format("csv") \
    .schema(skema_latihan) \
    .option("header", True) \
    .load(folder_latihan)

agregasi_latihan = df_stream_latihan.groupBy("kategori").agg(
    F.avg("pendapatan").alias("rata_rata_pendapatan"),
    F.count("trx_id").alias("jumlah_transaksi")
)

query_latihan = agregasi_latihan.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("latihan_agg") \
    .start()

print("Query latihan berjalan:", query_latihan.status)

Query latihan berjalan: {'message': 'Getting offsets from FileStreamSource[file:/mnt/alterra/repos/praktikum-big-data/pertemuan-10/latihan10_stream]', 'isDataAvailable': False, 'isTriggerActive': True}


### Latihan Mandiri — Soal 2: Ingestion Data Batch 1

In [12]:
pd.DataFrame({
    "trx_id": ["L1", "L2", "L3"],
    "kategori": ["Elektronik", "Makanan", "Elektronik"],
    "pendapatan": [200000, 50000, 400000]
}).to_csv(f"{folder_latihan}/batch_latihan_1.csv", index=False)

time.sleep(2)
query_latihan.processAllAvailable()

spark.sql("SELECT * FROM latihan_agg").show()

+----------+--------------------+----------------+
|  kategori|rata_rata_pendapatan|jumlah_transaksi|
+----------+--------------------+----------------+
|   Makanan|             50000.0|               1|
|Elektronik|            300000.0|               2|
+----------+--------------------+----------------+



### Latihan Mandiri — Soal 3: Ingestion Data Batch 2

In [13]:
pd.DataFrame({
    "trx_id": ["L4", "L5"],
    "kategori": ["Elektronik", "Makanan"],
    "pendapatan": [600000, 150000]
}).to_csv(f"{folder_latihan}/batch_latihan_2.csv", index=False)

time.sleep(2)
query_latihan.processAllAvailable()

spark.sql("SELECT * FROM latihan_agg").show()

+----------+--------------------+----------------+
|  kategori|rata_rata_pendapatan|jumlah_transaksi|
+----------+--------------------+----------------+
|   Makanan|            100000.0|               2|
|Elektronik|            400000.0|               3|
+----------+--------------------+----------------+



### Latihan Mandiri — Soal 4: Refleksi Skema Eksplisit vs Inferensi Skema

Spark Structured Streaming mewajibkan pendefinisian skema secara eksplisit (`StructType`) karena data streaming bersifat tidak terbatas (*unbounded*) dan terus mengalir secara inkremental dari waktu ke waktu. Tidak seperti pemrosesan batch yang dapat memindai seluruh berkas terlebih dahulu untuk menyimpulkan tipe kolom (`inferSchema`), sistem streaming tidak memiliki kondisi akhir untuk menunggu kelengkapan data. Definisi skema eksplisit menjamin kepastian kontrak data (*data contract*) sejak micro-batch pertama, mencegah kegagalan pipeline di tengah jalan, dan menjaga konsistensi state akumulasi data terdistribusi.

### Penutupan Sesi Latihan

In [14]:
query_latihan.stop()
print("Query latihan dihentikan. Status aktif:", query_latihan.isActive)

Query latihan dihentikan. Status aktif: False
